In [1]:
from pyspark.sql import functions as F

# 1. Odczyt surowej tabeli Delta z Lakehouse_Bronze
bronze_openaq_path = "abfss://2f59ccc2-e9d6-44b5-8c0c-1bdbf1fe148a@onelake.dfs.fabric.microsoft.com/33b93b7e-0899-43c6-81a8-654bc740aa32/Tables/dbo/openaq_locations_bronze"

df_raw = spark.read.format("delta").load(bronze_openaq_path)

# 2. Rozbicie tablicy sensorów i spłaszczenie zagnieżdżonych obiektów JSON
df_exploded = df_raw.withColumn("sensor", F.explode_outer("sensors"))

df_silver = (
    df_exploded
    .select(
        F.col("id").cast("long").alias("location_id"),
        F.trim(F.col("name")).alias("location_name"),
        F.col("locality"),
        F.col("timezone"),
        F.col("isMobile").cast("boolean").alias("is_mobile"),
        F.col("isMonitor").cast("boolean").alias("is_monitor"),
        F.col("coordinates.latitude").cast("double").alias("latitude"),
        F.col("coordinates.longitude").cast("double").alias("longitude"),
        F.col("country.id").cast("long").alias("country_id"),
        F.col("country.code").alias("country_code"),
        F.col("country.name").alias("country_name"),
        F.col("provider.name").alias("provider_name"),
        F.col("owner.name").alias("owner_name"),
        F.col("sensor.id").cast("long").alias("sensor_id"),
        F.col("sensor.name").alias("sensor_name"),
        F.col("sensor.parameter.name").alias("parameter_name"),
        F.col("sensor.parameter.displayName").alias("parameter_display_name"),
        F.col("sensor.parameter.units").alias("parameter_units")
    )
    .filter(
        F.col("location_id").isNotNull() &
        F.col("latitude").isNotNull() &
        F.col("longitude").isNotNull()
    )
    .dropDuplicates(["location_id", "sensor_id"])
)

# 3. Zapis oczyszczonej tabeli Delta do domyślnego Lakehouse_Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("openaq_silver")
)

print(f"Zapisano tabelę openaq_silver. Liczba rekordów: {df_silver.count()}")
display(df_silver.limit(10))

StatementMeta(, 88a89a00-fe86-4d37-b554-827764d3ec12, 3, Finished, Available, Finished, False)

Zapisano tabelę openaq_silver. Liczba rekordów: 395


SynapseWidget(Synapse.DataFrame, 5fc7b315-ed5c-4f51-bc55-094a961e6d25)